# Proyecto DS3021 - Parte 2: Adquisición de datos de Mercado Libre Perú


## 1. Instalación de librerías

Instala Selenium, BeautifulSoup, Requests y pandas.


In [1]:
# Ejecutar una sola vez en el entorno local.
%pip install -q selenium beautifulsoup4 requests pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Librerías y configuración

Define la fuente, las categorías, las pausas, los headers y las columnas del CSV crudo.


In [2]:
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

CATEGORIAS = [
    {'nombre': 'celulares', 'url': 'https://listado.mercadolibre.com.pe/celulares'},
    {'nombre': 'laptops', 'url': 'https://listado.mercadolibre.com.pe/laptops'},
    {'nombre': 'televisores', 'url': 'https://listado.mercadolibre.com.pe/televisores'},
    {'nombre': 'refrigeradoras', 'url': 'https://listado.mercadolibre.com.pe/refrigeradoras'},
    {'nombre': 'audifonos', 'url': 'https://listado.mercadolibre.com.pe/audifonos'},
]

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/152.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'es-PE,es;q=0.9',
}

PAUSA_SEGUNDOS = 10
MAX_PAGINAS = 20
RUTA_CSV = 'mercadolibre_raw.csv'
RUTA_DIAGNOSTICO = 'diagnostico_columnas.csv'
RUTA_DUPLICADOS = 'duplicados_detectados.csv'

COLUMNAS = [
    'categoria', 'pagina', 'posicion_pagina', 'id_publicacion',
    'titulo', 'moneda', 'precio_actual_texto',
    'precio_anterior_texto', 'cuotas_texto', 'vendedor_texto',
    'url_publicacion', 'url_fuente', 'fecha_extraccion'
]

print('Categorías:', len(CATEGORIAS))
print('Máximo de páginas por categoría:', MAX_PAGINAS)


Categorías: 5
Máximo de páginas por categoría: 20


## 3. Verificación de robots.txt

Comprueba que el archivo `robots.txt` de la fuente sea accesible.


In [3]:
try:
    respuesta_robots = requests.get(
        'https://www.mercadolibre.com.pe/robots.txt',
        headers=HEADERS,
        timeout=30
    )
    print('Estado de robots.txt:', respuesta_robots.status_code)
    if respuesta_robots.status_code == 200:
        print('robots.txt accesible. Las rutas fueron revisadas en la Parte 1.')
    else:
        print('No fue posible consultar robots.txt correctamente.')
except Exception as error:
    print('Error consultando robots.txt:', error)


Estado de robots.txt: 200
robots.txt accesible. Las rutas fueron revisadas en la Parte 1.


## 4. Extracción del HTML

Estas funciones identifican las tarjetas y conservan los textos originales y los valores ausentes.


In [4]:
def buscar_clase(raiz, clases):
    for clase in clases:
        elemento = raiz.find(class_=clase)
        if elemento is not None:
            return elemento
    return None


def obtener_texto(elemento):
    if elemento is None:
        return None
    texto = elemento.get_text(' ', strip=True)
    return texto if texto != '' else None


def detectar_tarjetas(soup):
    tarjetas = soup.find_all('li', class_='ui-search-layout__item')
    if len(tarjetas) == 0:
        tarjetas = soup.find_all('div', class_='ui-search-result__wrapper')
    if len(tarjetas) == 0:
        tarjetas = soup.find_all('div', class_='poly-card')
    return tarjetas


def extraer_id(texto):
    texto = (texto or '').upper()
    posicion = texto.find('MPE')
    while posicion != -1:
        indice = posicion + 3
        while indice < len(texto) and texto[indice] in '-_':
            indice += 1
        inicio = indice
        while indice < len(texto) and texto[indice].isdigit():
            indice += 1
        if indice > inicio:
            return 'MPE' + texto[inicio:indice]
        posicion = texto.find('MPE', posicion + 3)
    return None


def extraer_precio(elemento):
    if elemento is None:
        return None
    moneda = buscar_clase(elemento, ['andes-money-amount__currency-symbol'])
    entero = buscar_clase(elemento, ['andes-money-amount__fraction'])
    centavos = buscar_clase(elemento, ['andes-money-amount__cents'])
    if entero is not None:
        resultado = obtener_texto(entero)
        if centavos is not None:
            resultado = resultado + ',' + obtener_texto(centavos)
        if moneda is not None:
            resultado = obtener_texto(moneda) + ' ' + resultado
        return resultado
    return obtener_texto(elemento)


def procesar_html(html, categoria, pagina, url_fuente):
    soup = BeautifulSoup(html, 'html.parser')
    tarjetas = detectar_tarjetas(soup)
    filas = []

    for posicion, tarjeta in enumerate(tarjetas, start=1):
        titulo = buscar_clase(
            tarjeta,
            ['poly-component__title', 'ui-search-item__title']
        )
        enlace = titulo if titulo is not None and titulo.name == 'a' else tarjeta.find('a')
        url_publicacion = enlace.get('href') if enlace is not None else None

        precio_actual = buscar_clase(
            tarjeta,
            ['poly-price__current', 'ui-search-price__second-line']
        )
        precio_anterior = buscar_clase(
            tarjeta,
            ['andes-money-amount--previous', 'ui-search-price__original-value']
        )
        moneda = buscar_clase(
            precio_actual if precio_actual is not None else tarjeta,
            ['andes-money-amount__currency-symbol']
        )
        cuotas = buscar_clase(
            tarjeta,
            ['poly-price__installments', 'ui-search-installments']
        )
        vendedor = buscar_clase(
            tarjeta,
            ['poly-component__seller', 'ui-search-official-store-label']
        )

        filas.append({
            'categoria': categoria,
            'pagina': pagina,
            'posicion_pagina': posicion,
            'id_publicacion': extraer_id(
                (url_publicacion or '') + ' ' + str(tarjeta)
            ),
            'titulo': obtener_texto(titulo),
            'moneda': obtener_texto(moneda),
            'precio_actual_texto': extraer_precio(precio_actual),
            'precio_anterior_texto': extraer_precio(precio_anterior),
            'cuotas_texto': obtener_texto(cuotas),
            'vendedor_texto': obtener_texto(vendedor),
            'url_publicacion': url_publicacion,
            'url_fuente': url_fuente,
            'fecha_extraccion': time.strftime('%Y-%m-%d %H:%M:%S'),
        })

    return filas


## 5. Navegación con Selenium

Abre Chrome, permite la autenticación manual y avanza mediante el botón real `Siguiente`.


In [5]:
def crear_driver():
    opciones = Options()
    opciones.add_argument('--start-maximized')
    opciones.add_argument('--lang=es-PE')
    opciones.add_argument('--user-agent=' + HEADERS['User-Agent'])
    return webdriver.Chrome(options=opciones)


def requiere_autenticacion(url):
    url = (url or '').lower()
    return (
        'account-verification' in url
        or '/login/' in url
        or '/lgz/' in url
    )


def esperar_tarjetas(driver):
    for intento in range(10):
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        if len(detectar_tarjetas(soup)) > 0:
            return True
        if requiere_autenticacion(driver.current_url):
            return False
        time.sleep(2)
    return False


def revisar_autenticacion(driver, url_destino):
    if requiere_autenticacion(driver.current_url):
        print('\nACCIÓN MANUAL NECESARIA')
        print('Completa el inicio de sesión en la ventana de Chrome.')
        input('Cuando termine, vuelve al notebook y presiona ENTER: ')
        if requiere_autenticacion(driver.current_url):
            driver.get(url_destino)
        time.sleep(PAUSA_SEGUNDOS)


def avanzar_siguiente(driver):
    cuerpo = driver.find_element(By.TAG_NAME, 'body')
    for intento in range(3):
        cuerpo.send_keys(Keys.END)
        time.sleep(2)

    selectores = [
        'li.andes-pagination__button--next a',
        '.andes-pagination__button--next a',
        'a[title="Siguiente"]',
        'a[aria-label="Siguiente"]',
        'button[aria-label="Siguiente"]',
    ]

    botones = []
    for selector in selectores:
        botones.extend(driver.find_elements(By.CSS_SELECTOR, selector))

    if len(botones) == 0:
        for elemento in driver.find_elements(By.CSS_SELECTOR, 'a, button'):
            texto = (elemento.text or '').strip().lower()
            titulo = (elemento.get_attribute('title') or '').lower()
            etiqueta = (elemento.get_attribute('aria-label') or '').lower()
            if 'siguiente' in texto or 'siguiente' in titulo or 'siguiente' in etiqueta:
                botones.append(elemento)

    for boton in botones:
        try:
            url_anterior = driver.current_url
            html_anterior = driver.page_source
            boton.click()
            time.sleep(PAUSA_SEGUNDOS)
            revisar_autenticacion(driver, url_anterior)
            if driver.current_url != url_anterior or driver.page_source != html_anterior:
                return True
        except Exception:
            continue

    return False


## 6. Scraping y guardado incremental

Recorre hasta veinte páginas por categoría y agrega cada página al CSV sin limpiar nulos ni duplicados.


In [6]:
def ejecutar_scraper():
    driver = crear_driver()
    primera_pagina_guardada = True
    total = 0

    try:
        for categoria in CATEGORIAS:
            driver.get(categoria['url'])
            time.sleep(PAUSA_SEGUNDOS)
            revisar_autenticacion(driver, categoria['url'])
            paginas_vistas = set()

            for pagina in range(1, MAX_PAGINAS + 1):
                try:
                    if not esperar_tarjetas(driver):
                        revisar_autenticacion(driver, driver.current_url)

                    html = driver.page_source
                    url_actual = driver.current_url
                    filas = procesar_html(
                        html,
                        categoria['nombre'],
                        pagina,
                        url_actual
                    )

                    if len(filas) == 0:
                        print(f"[{categoria['nombre']}] página {pagina}: sin registros")
                        break

                    firma_pagina = tuple(
                        (fila['id_publicacion'], fila['titulo'], fila['url_publicacion'])
                        for fila in filas
                    )
                    if firma_pagina in paginas_vistas:
                        print(f"[{categoria['nombre']}] página repetida; fin de categoría")
                        break
                    paginas_vistas.add(firma_pagina)

                    df_pagina = pd.DataFrame(filas, columns=COLUMNAS)
                    modo_guardado = 'w' if primera_pagina_guardada else 'a'
                    df_pagina.to_csv(
                        RUTA_CSV,
                        mode=modo_guardado,
                        header=primera_pagina_guardada,
                        index=False,
                        encoding='utf-8-sig'
                    )
                    primera_pagina_guardada = False
                    total += len(df_pagina)

                    print(
                        f"[{categoria['nombre']}] página {pagina}: "
                        f"{len(df_pagina)} guardados | acumulado: {total}"
                    )
                    print('URL:', url_actual)

                    if pagina < MAX_PAGINAS:
                        if not avanzar_siguiente(driver):
                            print('No existe una página siguiente. Fin de categoría.')
                            break

                except Exception as error:
                    print(
                        f"Error en {categoria['nombre']}, página {pagina}: {error}"
                    )
                    break

    finally:
        driver.quit()

    if total == 0:
        raise RuntimeError('No se extrajeron registros.')

    print('Extracción terminada. Registros crudos:', total)
    return pd.read_csv(RUTA_CSV)


## 7. Ejecución

Ejecuta el scraper y muestra una muestra del dataset crudo.


In [7]:
df_final = ejecutar_scraper()
print('Dimensiones:', df_final.shape)
display(df_final.head(10))


[celulares] página 1: 48 guardados | acumulado: 48
URL: https://listado.mercadolibre.com.pe/celulares
[celulares] página 2: 48 guardados | acumulado: 96
URL: https://listado.mercadolibre.com.pe/celulares-telefonos/celulares-smartphones/celulares_Desde_49_NoIndex_True
[celulares] página 3: 48 guardados | acumulado: 144
URL: https://listado.mercadolibre.com.pe/celulares-telefonos/celulares-smartphones/celulares_Desde_97_NoIndex_True
[celulares] página 4: 48 guardados | acumulado: 192
URL: https://listado.mercadolibre.com.pe/celulares-telefonos/celulares-smartphones/celulares_Desde_145_NoIndex_True
[celulares] página 5: 48 guardados | acumulado: 240
URL: https://listado.mercadolibre.com.pe/celulares-telefonos/celulares-smartphones/celulares_Desde_193_NoIndex_True
[celulares] página 6: 48 guardados | acumulado: 288
URL: https://listado.mercadolibre.com.pe/celulares-telefonos/celulares-smartphones/celulares_Desde_241_NoIndex_True
[celulares] página 7: 48 guardados | acumulado: 336
URL: http

,categoria,pagina,posicion_pagina,id_publicacion,titulo,moneda,precio_actual_texto,precio_anterior_texto,cuotas_texto,vendedor_texto,url_publicacion,url_fuente,fecha_extraccion
0,celulares,1,1,MPE55308605,Apple iPhone 17 Pro Max (256 GB) - Color plata,S/,S/ 4.849,NaN,"6 cuotas de S/ 808 , 17 sin interés",NaN,https://www.mercadolibre.com.pe/apple-iphone-1...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
1,celulares,1,2,MPE65503965,Celular Samsung Galaxy S26 Ultra 512gb,S/,S/ 4.129,S/ 6.899,"6 cuotas de S/ 688 , 21 sin interés",NaN,https://www.mercadolibre.com.pe/celular-samsun...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
2,celulares,1,3,MPE55308632,Apple iPhone 17 Pro Max (256 GB) - Azul profundo,S/,S/ 4.849,NaN,"6 cuotas de S/ 808 , 17 sin interés",NaN,https://www.mercadolibre.com.pe/apple-iphone-1...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
3,celulares,1,4,MPE55308781,Apple iPhone 17 Pro (256 GB) - Azul profundo,S/,S/ 4.479,S/ 6.000,"6 cuotas de S/ 746 , 50 sin interés",NaN,https://www.mercadolibre.com.pe/apple-iphone-1...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
4,celulares,1,5,MPE47797906,Samsung Galaxy A56 5G Dual Sim 256GB 12GB Negr...,S/,S/ 1.517,S/ 2.199,"6 cuotas de S/ 252 , 88 sin interés",NaN,https://www.mercadolibre.com.pe/samsung-galaxy...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
5,celulares,1,6,MPE47797574,Cel Samsung A56 12 Gb Ram 256 Gb 5g 6.7 Pulgad...,S/,S/ 1.464,S/ 2.449,"6 cuotas de S/ 244 , 04 sin interés",NaN,https://www.mercadolibre.com.pe/cel-samsung-a5...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
6,celulares,1,7,MPE50678684,Motorola Edge 60 Fusion 8GB RAM 256GB ROM azul...,S/,"S/ 987,90",S/ 1.500,"6 cuotas de S/ 164 , 65 sin interés",NaN,https://www.mercadolibre.com.pe/motorola-edge-...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
7,celulares,1,8,MPE48954564,Samsung A56 5G 5G Dual SIM 256 GB gris 12 GB RAM,S/,S/ 1.464,S/ 2.449,"6 cuotas de S/ 244 , 04 sin interés",NaN,https://www.mercadolibre.com.pe/samsung-a56-5g...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
8,celulares,1,9,MPE45513937,Samsung Galaxy S25 Ultra 512gb Titanium Black,S/,S/ 3.339,S/ 6.999,"6 cuotas de S/ 556 , 54 sin interés",NaN,https://www.mercadolibre.com.pe/samsung-galaxy...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05
9,celulares,1,10,MPE67588045,Xiaomi Pocophone Poco X8 Pro Negro 12+512 GB,S/,S/ 1.599,S/ 2.500,"6 cuotas de S/ 266 , 50 sin interés",NaN,https://www.mercadolibre.com.pe/xiaomi-pocopho...,https://listado.mercadolibre.com.pe/celulares,2026-09-13 19:54:05


## 8. Diagnóstico y validación

Reporta nulos y duplicados sin modificar el CSV crudo y verifica los requisitos mínimos.


In [8]:
diagnostico = pd.DataFrame({
    'columna': df_final.columns,
    'nulos': [df_final[columna].isnull().sum() for columna in df_final.columns],
    'porcentaje_nulos': [
        round(df_final[columna].isnull().mean() * 100, 2)
        for columna in df_final.columns
    ],
    'valores_unicos': [
        df_final[columna].nunique(dropna=True)
        for columna in df_final.columns
    ],
})

duplicados = df_final[
    df_final['id_publicacion'].notnull()
    & df_final.duplicated('id_publicacion', keep=False)
]

diagnostico.to_csv(RUTA_DIAGNOSTICO, index=False, encoding='utf-8-sig')
duplicados.to_csv(RUTA_DUPLICADOS, index=False, encoding='utf-8-sig')

print('Registros crudos:', len(df_final))
print('Columnas:', len(df_final.columns))
print('IDs únicos:', df_final['id_publicacion'].nunique(dropna=True))
print('Filas con ID repetido:', len(duplicados))
print('Cumple mínimo de 1000 registros:', len(df_final) >= 1000)
print('Cumple mínimo de 6 atributos:', len(df_final.columns) >= 6)
print('CSV crudo:', RUTA_CSV)
print('Diagnóstico:', RUTA_DIAGNOSTICO)

display(df_final['categoria'].value_counts(dropna=False))
display(diagnostico.sort_values('porcentaje_nulos', ascending=False))


Registros crudos: 5080
Columnas: 13
IDs únicos: 4542
Filas con ID repetido: 685
Cumple mínimo de 1000 registros: True
Cumple mínimo de 6 atributos: True
CSV crudo: mercadolibre_raw.csv
Diagnóstico: diagnostico_columnas.csv


categoria
televisores       1200
audifonos         1200
laptops           1040
celulares          960
refrigeradoras     680
Name: count, dtype: int64

,columna,nulos,porcentaje_nulos,valores_unicos
9,vendedor_texto,3255,64.07,107
7,precio_anterior_texto,2036,40.08,942
8,cuotas_texto,106,2.09,2337
1,pagina,0,0.00,20
0,categoria,0,0.00,5
4,titulo,0,0.00,4249
3,id_publicacion,0,0.00,4542
2,posicion_pagina,0,0.00,60
6,precio_actual_texto,0,0.00,1969
5,moneda,0,0.00,1


## Resultado

El CSV generado conserva los valores ausentes y los posibles duplicados para la etapa de limpieza de datos.
